# Baseline evaluation: Ollama chat only (no rules, no KG)

This notebook replays the project’s fever triage scenarios as a **pure conversational LLM baseline** using **Ollama** (Qwen).

- **No PyDatalog rules**
- **No Neo4j / SNOMED KG retrieval**

Goal: see whether “just chatting” produces a clinically-safe, bounded triage-style response.

**Disclaimer:** course prototype / educational use only — not medical advice.

In [1]:
import os
from pathlib import Path
import pandas as pd

# ---- Ollama model selection ----
# Pick any Qwen model you have installed in Ollama.
# Examples:
#   ollama pull qwen2.5:7b-instruct
#   ollama pull qwen2.5:14b-instruct
os.environ.setdefault("OLLAMA_MODEL", "qwen2.5:7b-instruct")
os.environ.setdefault("OLLAMA_BASE_URL", "http://127.0.0.1:11434")

OLLAMA_MODEL = os.environ["OLLAMA_MODEL"]
OLLAMA_BASE_URL = os.environ["OLLAMA_BASE_URL"]

print("OLLAMA_MODEL=", OLLAMA_MODEL)
print("OLLAMA_BASE_URL=", OLLAMA_BASE_URL)

# Scenario CSV used by CareTrace harness (same turns_json)
SCENARIOS_CSV = Path("caretrace/evaluation/scenarios.csv")
assert SCENARIOS_CSV.exists(), f"Not found: {SCENARIOS_CSV.resolve()}"

OLLAMA_MODEL= qwen2.5:7b-instruct
OLLAMA_BASE_URL= http://127.0.0.1:11434


In [2]:
import sys
print("Python:", sys.executable)

# Installs into the **current kernel** (should be conda env neurosymbolic_ai)
%pip install -q "langchain-ollama>=0.2.0"


Python: /opt/miniconda3/envs/neurosymbolic_ai/bin/python3.11
Note: you may need to restart the kernel to use updated packages.


In [3]:
df = pd.read_csv(SCENARIOS_CSV)
df[["id", "expected_disposition", "description"]]

,id,expected_disposition,description
0,scenario_1_home,HOME_MANAGEMENT,Docs/Scenario.txt — home with safety netting
1,scenario_2_er,ER_NOW,Docs/Scenario.txt — ER (adds explicit normal b...
2,urgent_repeated_vomit,URGENT_SAME_DAY,Repeated vomiting + poor fluids → urgent (not ...


In [4]:
import json

def load_turns(scenario_id: str) -> list[str]:
    row = df.loc[df["id"] == scenario_id]
    assert len(row) == 1, f"Scenario not found: {scenario_id}" 
    turns_json = row.iloc[0]["turns_json"]
    turns = json.loads(turns_json)
    assert isinstance(turns, list) and all(isinstance(t, str) for t in turns)
    return turns


def format_transcript(turns: list[str]) -> str:
    lines = []
    for i, t in enumerate(turns, start=1):
        lines.append(f"Turn {i} (caregiver): {t}")
    return "\n".join(lines)


scenario_id = "scenario_1_home"  # change to scenario_2_er, urgent_repeated_vomit
turns = load_turns(scenario_id)
print(format_transcript(turns))

Turn 1 (caregiver): My 6-year-old has a fever, threw up once, and looks really wiped out.
Turn 2 (caregiver): Temp is 101.8. He's tired but answers me. No breathing issues. He's sipping water, not much though. He's been on medication for a recent ear infection.
Turn 3 (caregiver): He's on amoxicillin. Last dose was earlier tonight. Just vomited once. He peed earlier this evening.


In [5]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model=OLLAMA_MODEL,
    base_url=OLLAMA_BASE_URL,
    temperature=0.3,
)

SYSTEM_PROMPT = """You are a cautious pediatric after-hours triage assistant for worried caregivers.

Constraints:
- Give a clinically conservative response.
- Ask only essential follow-up questions (3–7) if needed.
- Include explicit red-flag escalation criteria (when to go to ER now).
- Do not invent vitals or history not provided.
- Do not provide exact medication dosing without weight and product concentration.
- Keep the tone calm and direct.

Output format:
1) Summary of what you heard
2) What to do now
3) What to watch for (go-now triggers)
4) Questions you still need (if any)
"""

transcript = format_transcript(turns)

resp = llm.invoke(
    [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": transcript},
    ]
)

print(resp.content)

/opt/miniconda3/envs/neurosymbolic_ai/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1) Summary of what I heard:
- Your 6-year-old has a fever of 101.8°F, appears tired, and has vomited once.
- He is otherwise alert and responsive to your questions.
- He recently had an ear infection and was prescribed amoxicillin; the last dose was earlier today.
- He has urinated at least once in the past few hours.

2) What to do now:
- Monitor his temperature every 4 hours. Use a rectal thermometer for accuracy, especially if he's younger than 3 years old.
- Ensure he stays hydrated by offering small sips of clear fluids like water or Pedialyte.
- Keep him comfortable with light clothing and a cool environment.

3) What to watch for (go-now triggers):
- Persistent vomiting
- Difficulty breathing
- Severe lethargy or unresponsiveness
- New or worsening symptoms such as rash, severe headache, stiff neck, or difficulty walking
- No urination in 8 hours

4) Questions I still need:
- Has he had any diarrhea?
- How long has the fever been present?


In [ ]:
# Optional: run CareTrace (rules + must-ask gating) for side-by-side comparison
import os

os.environ.setdefault("CARETRACE_MOCK_LLM", "1")
os.environ.setdefault("CARETRACE_SKIP_NEO4J", "1")

from caretrace.evaluation.harness import replay_turns
from caretrace.pretty_output import display_caretrace_turn

final_state = replay_turns(turns)
display_caretrace_turn(
    final_state.get("decision"),
    final_state.get("assistant_reply"),
    # max_reply_chars=600,  # uncomment to cap length like the old preview
)